# Task 1: EDA & Time Series Property Analysis — Brent Oil Prices

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
import ruptures as rpt
import warnings
warnings.filterwarnings('ignore')

## 1. Load & Preprocess Data

In [ ]:
df = pd.read_csv('../data/BrentOilPrices.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')
df = df.sort_values('Date').set_index('Date')
df = df.asfreq('D').ffill()  # forward-fill non-trading days
df['log_return'] = np.log(df['Price'] / df['Price'].shift(1))
print(df.shape)
df.head()

## 2. Load Key Events

In [ ]:
events = pd.read_csv('../data/key_events.csv', parse_dates=['date'])
events

## 3. Price Series with Event Annotations

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df.index, df['Price'], linewidth=0.8, color='steelblue', label='Brent Price (USD/bbl)')

for _, row in events.iterrows():
    if row['date'] >= df.index.min() and row['date'] <= df.index.max():
        ax.axvline(row['date'], color='red', alpha=0.4, linewidth=0.8, linestyle='--')

ax.set_title('Brent Crude Oil Price (1987–2022) with Key Events')
ax.set_ylabel('Price (USD/bbl)')
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.savefig('../data/price_with_events.png', dpi=150)
plt.show()

## 4. Trend & Volatility Analysis

In [ ]:
df['roll_mean_90'] = df['Price'].rolling(90).mean()
df['roll_std_90'] = df['Price'].rolling(90).std()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(df.index, df['Price'], linewidth=0.6, alpha=0.7, label='Price')
axes[0].plot(df.index, df['roll_mean_90'], linewidth=1.5, color='orange', label='90-day Rolling Mean')
axes[0].set_ylabel('Price (USD/bbl)')
axes[0].legend()
axes[0].set_title('Price Trend')

axes[1].plot(df.index, df['roll_std_90'], linewidth=0.8, color='crimson')
axes[1].set_ylabel('Rolling Std Dev')
axes[1].set_title('Volatility (90-day Rolling Std Dev)')

plt.tight_layout()
plt.savefig('../data/trend_volatility.png', dpi=150)
plt.show()

## 5. Stationarity Tests

In [ ]:
def run_stationarity_tests(series, name):
    clean = series.dropna()
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(clean)
    kpss_stat, kpss_p, _, kpss_crit = kpss(clean, regression='c', nlags='auto')
    print(f"--- {name} ---")
    print(f"ADF  stat={adf_stat:.4f}, p={adf_p:.4f}  → {'Stationary' if adf_p < 0.05 else 'Non-stationary'}")
    print(f"KPSS stat={kpss_stat:.4f}, p={kpss_p:.4f}  → {'Non-stationary' if kpss_p < 0.05 else 'Stationary'}\n")

run_stationarity_tests(df['Price'], 'Price Levels')
run_stationarity_tests(df['log_return'], 'Log Returns')

## 6. Change Point Detection

**Purpose**: Change point models detect dates where the statistical properties (mean, variance) of the time series shift significantly. In oil price analysis, these structural breaks often correspond to major market regime changes — e.g., a sustained price crash or a new high-price era. The PELT (Pruned Exact Linear Time) algorithm efficiently finds the optimal number and location of breakpoints by minimizing a penalized cost function.

**Expected outputs**: Dates of detected change points, and the mean/std of each segment between breakpoints.

**Limitations**: The algorithm cannot tell *why* a break occurred — only *when*. The penalty parameter controls sensitivity; too low produces spurious breaks, too high misses real ones.

In [ ]:
price_array = df['Price'].values
algo = rpt.Pelt(model='rbf').fit(price_array)
breakpoints = algo.predict(pen=10)

bp_dates = [df.index[i - 1] for i in breakpoints[:-1]]  # exclude last sentinel
print(f"Detected {len(bp_dates)} change points:")
for d in bp_dates:
    print(f"  {d.date()}")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df.index, df['Price'], linewidth=0.8, color='steelblue')

for d in bp_dates:
    ax.axvline(d, color='green', linewidth=1.2, linestyle='--', alpha=0.8)

# Annotate segment means
prev = 0
for bp in breakpoints:
    seg = df['Price'].iloc[prev:bp]
    seg_mean = seg.mean()
    ax.hlines(seg_mean, df.index[prev], df.index[bp - 1], colors='orange', linewidth=1.5)
    prev = bp

ax.set_title('Brent Oil Price — PELT Change Points (green) & Segment Means (orange)')
ax.set_ylabel('Price (USD/bbl)')
plt.tight_layout()
plt.savefig('../data/change_points.png', dpi=150)
plt.show()

## 7. Segment Statistics Summary

In [ ]:
segments = []
prev = 0
for bp in breakpoints:
    seg = df['Price'].iloc[prev:bp]
    segments.append({
        'start': df.index[prev].date(),
        'end': df.index[bp - 1].date(),
        'mean_price': round(seg.mean(), 2),
        'std_price': round(seg.std(), 2),
        'n_days': len(seg)
    })
    prev = bp

pd.DataFrame(segments)